# MB1 — Semantic Moment Probe Benchmark: Mode A

Prepare chronological raw-video contact sheets for later GPT-5.6 Sol interval annotation. This notebook performs no semantic labeling and no model inference.

## Kaggle inputs

1. Raw AIC dataset: `/kaggle/input/datasets/nadkli/dataset-aic`
2. RT2 anchor identities: `/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle`

Stage1, CLIP, OPUS, vectors, OCR, objects, and Event Graph assets are not inputs. Output is zipped to `/kaggle/working/triage_eg_mb1_candidates.zip`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print("resolved commit:", COMMIT)

In [ ]:
DATASET_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
BENCHMARK_INPUT = Path(os.environ.get("AIC_MB1_BENCHMARK_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle"))
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_mb1_candidates")
ZIP_PATH = Path("/kaggle/working/triage_eg_mb1_candidates.zip")
print({"raw_dataset_input": str(DATASET_INPUT), "rt2_benchmark_input": str(BENCHMARK_INPUT), "output": str(OUTPUT_ROOT), "zip": str(ZIP_PATH)})

In [ ]:
def resolve_input_file(root: Path, filename: str, search_root: Path = Path("/kaggle/input")) -> Path:
    root = Path(root)
    if root.is_file() and root.name == filename:
        return root.resolve()
    matches = sorted(root.rglob(filename)) if root.is_dir() else []
    if not matches and search_root.is_dir():
        matches = sorted(search_root.rglob(filename))
    matches = sorted({path.resolve() for path in matches if path.is_file()})
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {filename}; found {matches}")
    return matches[0]

def resolve_dataset_root(requested: Path, probe_video: str = "L23_V005.mp4") -> Path:
    requested = Path(requested)
    direct = requested / "Videos_L23" / "video" / probe_video
    if direct.is_file():
        return requested.resolve()
    roots = []
    for base in (requested, Path("/kaggle/input")):
        if base.is_dir():
            roots.extend(path.parents[2].resolve() for path in base.rglob(probe_video) if path.parent.name == "video")
    roots = sorted(set(roots))
    if len(roots) != 1:
        raise RuntimeError(f"Expected one raw dataset root; found {roots}")
    return roots[0]

DATASET_ROOT = resolve_dataset_root(DATASET_INPUT)
BENCHMARK_PATH = resolve_input_file(BENCHMARK_INPUT, "rt2_ai_benchmark.jsonl")
print({"resolved_raw_dataset": str(DATASET_ROOT), "resolved_rt2_benchmark": str(BENCHMARK_PATH)})

In [ ]:
from triage_eg.experiments.mb1 import MB1Settings, preflight_mb1

SETTINGS = MB1Settings()
PREFLIGHT = preflight_mb1(DATASET_ROOT, BENCHMARK_PATH, SETTINGS)
print(json.dumps(PREFLIGHT, indent=2))

In [ ]:
from triage_eg.experiments.mb1 import prepare_mb1_candidates

working_root = Path("/kaggle/working").resolve()
resolved_output = OUTPUT_ROOT.resolve()
if resolved_output.parent != working_root or resolved_output.name != "triage_eg_mb1_candidates":
    raise RuntimeError(f"Unsafe MB1 output path: {resolved_output}")
if resolved_output.exists():
    shutil.rmtree(resolved_output)
ZIP_PATH.unlink(missing_ok=True)
RESULT = prepare_mb1_candidates(DATASET_ROOT, BENCHMARK_PATH, OUTPUT_ROOT, settings=SETTINGS, build_git_commit=COMMIT)
print(json.dumps(RESULT, indent=2))

In [ ]:
from IPython.display import Image, display

rows = [json.loads(line) for line in (OUTPUT_ROOT / "mb1_candidate_manifest.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
assert 24 <= len(rows) <= 30
assert all(row["displayed_frames"] == sorted(set(row["displayed_frames"])) for row in rows)
assert all((OUTPUT_ROOT / path).is_file() for row in rows for path in row["image_sheet_paths"])
print("candidate windows:", len(rows), "contact sheets:", sum(len(row["image_sheet_paths"]) for row in rows))
for path in rows[0]["image_sheet_paths"]:
    display(Image(filename=str(OUTPUT_ROOT / path)))

In [ ]:
from triage_eg.experiments.mb1 import create_mb1_bundle

archive = create_mb1_bundle(OUTPUT_ROOT, ZIP_PATH)
with __import__("zipfile").ZipFile(archive) as stream:
    members = stream.namelist()
assert not any(name.endswith((".pt", ".pth", ".bin", ".npy", ".npz", ".mp4")) for name in members)
print("DOWNLOAD ZIP:", archive, "size_bytes=", archive.stat().st_size, "members=", len(members))
print("MB1_MODE_A_IMPLEMENTATION_STATUS = COMPLETE")
print("MB1_CANDIDATE_PACK_STATUS = READY")
print("MB1_AI_ANNOTATION_STATUS = WAITING_FOR_AI")
print("M2_IMPLEMENTATION_STATUS = NOT_STARTED")